## 1. Load NHANES and score 3-year risk

Load the analysis file from `preprocessing.py` and add the CKD-PC 3-year risk of a ≥40% eGFR decline or kidney failure for every participant. Each person is scored with their own stratum (eGFR ≥60 or <60, with or without diabetes).

In [ ]:
import pandas as pd
from egfr40 import add_egfr40_risk_col

# Analysis-ready file from preprocessing.py
df = pd.read_parquet("../data/processed/nhanes_egfr40.parquet")

# Score every participant; column names already match the defaults
df = add_egfr40_risk_col(df)

# Label each participant's stratum for later summaries
df["stratum"] = (
    df["dm"].map({0: "No DM", 1: "DM"})
    + ", eGFR "
    + df["eGFR"].ge(60).map({True: ">=60", False: "<60"})
)

print(f"{len(df):,} participants, {df['egfr40_3year'].isna().sum()} missing scores")
df.groupby("stratum")["egfr40_3year"].describe().round(3)

## 1b. One risk score per participant

Every NHANES participant (one row per `SEQN`) now has an `egfr40_3year` score.
Show the first rows, then the ten highest-risk participants with the inputs that
drove their scores.

In [ ]:
# Columns worth seeing next to each participant's score
cols = ["SEQN", "stratum", "age", "sex", "eGFR", "uACR", "sbp", "bp_meds",
        "hf", "chd", "smoking", "bmi", "hba1c", "egfr40_3year"]

# First rows: one score per participant
display(df[cols].head(10).round(3))

# Highest-risk participants
display(df.sort_values("egfr40_3year", ascending=False)[cols].head(10).round(3))

## 2. Survey-weighted risk by stratum

NHANES oversamples some groups, so raw sample summaries don't describe the US population. Each participant's MEC exam weight (`WTMECPRP`) says how many US adults they represent. Weighting gives:

- **% of US adults** in each stratum
- **Weighted mean and median** 3-year risk
- **% at ≥10% risk**, a simple high-risk cutoff

These are complete-case estimates: weights are not re-adjusted for participants dropped for missing data, so treat population totals as approximate.

In [ ]:
import numpy as np


def weighted_quantile(values, weights, q):
    """Quantile q (0-1) of values, where each value counts by its weight."""
    order = np.argsort(values)
    v, w = np.asarray(values)[order], np.asarray(weights)[order]
    cum = np.cumsum(w) / w.sum()  # running share of total weight
    return v[np.searchsorted(cum, q)]


def weighted_summary(g):
    w, r = g["WTMECPRP"], g["egfr40_3year"]
    return pd.Series({
        "n": len(g),
        "us_adults_millions": w.sum() / 1e6,
        "mean_risk": np.average(r, weights=w),
        "median_risk": weighted_quantile(r, w, 0.5),
        "p90_risk": weighted_quantile(r, w, 0.9),
        "pct_risk_ge_10": np.average(r >= 0.10, weights=w) * 100,
    })


# Drop the few participants with zero weight (not examined)
dfw = df[df["WTMECPRP"] > 0]

by_stratum = dfw.groupby("stratum").apply(weighted_summary, include_groups=False)
overall = weighted_summary(dfw).rename("All adults")
summary = pd.concat([by_stratum, overall.to_frame().T])
summary["pct_us_adults"] = summary["us_adults_millions"] / overall["us_adults_millions"] * 100

summary.round({"us_adults_millions": 1, "mean_risk": 3, "median_risk": 3,
               "p90_risk": 3, "pct_risk_ge_10": 1, "pct_us_adults": 1})